# A3.3.2: SQL queries between tables

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* Core for SL and HL. Command term: **construct**. Builds on A3.3.1.

*`SELECT` can do much more once two tables are involved: combining them, filtering what comes back, matching text patterns and putting the results in a sensible order.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-2

In [11]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

In [12]:
# Install the SQL add-on into THIS kernel if it is missing (one time only). (Given.)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sql") is None:
    print("Installing the SQL add-on (one time, about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"])
    importlib.invalidate_caches()

import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE Artist (ArtistID TEXT PRIMARY KEY, ArtistName TEXT, Genre TEXT);
CREATE TABLE Concert (ConcertID TEXT PRIMARY KEY, ArtistID TEXT, VenueName TEXT, TicketPrice REAL, ConcertDate TEXT,
                      FOREIGN KEY (ArtistID) REFERENCES Artist (ArtistID));
INSERT INTO Artist VALUES ('A1', 'Dana Voss', 'Indie'), ('A2', 'Dana Marsh', 'Indie'),
                          ('A3', 'The Cobalt Keys', 'Jazz'), ('A4', 'Reo Vantage', 'Rock');
INSERT INTO Concert VALUES ('C1', 'A1', 'Riverside Hall', 45, '2027-03-04'),
                           ('C2', 'A2', 'Riverside Hall', 60, '2027-03-11'),
                           ('C3', 'A3', 'Blue Note Club', 35, '2027-04-02'),
                           ('C4', 'A4', 'Grand Arena', 90, '2027-04-20'),
                           ('C5', 'A1', 'Blue Note Club', 50, '2027-05-01');
""");


%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Two tables, linked by ArtistID

`Concert.ArtistID` is a **foreign key** that matches `Artist.ArtistID`, the primary key. Look at both tables.

In [13]:
%%sql
SELECT * FROM Artist;

ArtistID,ArtistName,Genre
A1,Dana Voss,Indie
A2,Dana Marsh,Indie
A3,The Cobalt Keys,Jazz
A4,Reo Vantage,Rock


In [14]:
%%sql
SELECT * FROM Concert;

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C1,A1,Riverside Hall,45.0,2027-03-04
C2,A2,Riverside Hall,60.0,2027-03-11
C3,A3,Blue Note Club,35.0,2027-04-02
C4,A4,Grand Arena,90.0,2027-04-20
C5,A1,Blue Note Club,50.0,2027-05-01


## 2. The command toolkit

**JOIN** combines rows from two tables where a matching column agrees. Without it, a query only ever sees one table. Every column that exists in both tables (here `ArtistID`) must be **qualified** with its table name, such as `Concert.ArtistID`.

In [15]:
%%sql
SELECT Artist.ArtistName, Artist.Genre, Concert.VenueName, Concert.TicketPrice
FROM Concert
JOIN Artist ON Concert.ArtistID = Artist.ArtistID;

ArtistName,Genre,VenueName,TicketPrice
Dana Voss,Indie,Riverside Hall,45.0
Dana Marsh,Indie,Riverside Hall,60.0
The Cobalt Keys,Jazz,Blue Note Club,35.0
Reo Vantage,Rock,Grand Arena,90.0
Dana Voss,Indie,Blue Note Club,50.0


**DISTINCT** removes repeated values.

In [16]:
%%sql
SELECT DISTINCT Genre FROM Artist;

Genre
Indie
Jazz
Rock


**WHERE** keeps only the rows that match a condition. Comparisons like `=`, `>`, `BETWEEN` (inclusive of both ends) and `LIKE` with the wildcard `%` are all conditions. Combine them with `AND`, `OR` and `NOT`.

In [17]:
%%sql
SELECT * FROM Concert WHERE VenueName = 'Blue Note Club';

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C3,A3,Blue Note Club,35.0,2027-04-02
C5,A1,Blue Note Club,50.0,2027-05-01


In [18]:
%%sql
SELECT * FROM Concert WHERE TicketPrice BETWEEN 40 AND 80;

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C1,A1,Riverside Hall,45.0,2027-03-04
C2,A2,Riverside Hall,60.0,2027-03-11
C5,A1,Blue Note Club,50.0,2027-05-01


In [19]:
%%sql
SELECT * FROM Concert WHERE VenueName LIKE '%Club';

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C3,A3,Blue Note Club,35.0,2027-04-02
C5,A1,Blue Note Club,50.0,2027-05-01


In [20]:
%%sql
SELECT * FROM Concert WHERE TicketPrice > 50 AND NOT VenueName = 'Grand Arena';

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C2,A2,Riverside Hall,60.0,2027-03-11


**ORDER BY** sorts the result (`DESC` for biggest first). **GROUP BY** collapses rows into piles and **HAVING** filters the piles.

In [21]:
%%sql
SELECT * FROM Concert ORDER BY TicketPrice DESC;

ConcertID,ArtistID,VenueName,TicketPrice,ConcertDate
C4,A4,Grand Arena,90.0,2027-04-20
C2,A2,Riverside Hall,60.0,2027-03-11
C5,A1,Blue Note Club,50.0,2027-05-01
C1,A1,Riverside Hall,45.0,2027-03-04
C3,A3,Blue Note Club,35.0,2027-04-02


In [22]:
%%sql
SELECT VenueName, COUNT(*) AS NumConcerts
FROM Concert
GROUP BY VenueName;

VenueName,NumConcerts
Blue Note Club,2
Grand Arena,1
Riverside Hall,2


In [23]:
%%sql
SELECT VenueName, COUNT(*) AS NumConcerts
FROM Concert
GROUP BY VenueName
HAVING COUNT(*) > 1;

VenueName,NumConcerts
Blue Note Club,2
Riverside Hall,2


> **WHERE filters ROWS before any grouping. HAVING filters GROUPS after `GROUP BY` has run.** Only HAVING can test an aggregate like `COUNT(*)`.

## 3. Try breaking things

Each of these is a real exam error. Run the cell and read what the database says.

In [24]:
def try_sql(text):
    try:
        print(db.execute(text).fetchall())
    except Exception as e:
        print("Error:", e)

try_sql("SELECT ArtistID FROM Concert JOIN Artist ON Concert.ArtistID = Artist.ArtistID")   # which ArtistID?
try_sql("SELECT VenueName, COUNT(*) FROM Concert WHERE COUNT(*) > 1 GROUP BY VenueName")    # aggregate in WHERE
try_sql("SELECT ArtistName FROM Artist WHERE ArtistName LIKE 'Dana%'")                       # works
try_sql("SELECT ArtistName FROM Artist WHERE ArtistName LIKE '%s'")                          # ends with s

Error: ambiguous column name: ArtistID
Error: misuse of aggregate: COUNT()
[('Dana Voss',), ('Dana Marsh',)]
[('Dana Voss',), ('The Cobalt Keys',)]


## 4. Your turn: construct the queries

Each TODO is a `%%sql name <<` cell. Select **exactly** the columns asked for, in that order, so the check can compare.

**TODO 1.** List every artist's name next to the venue they play (`ArtistName`, `VenueName`). Use a JOIN.

In [38]:
%%sql q1 <<
-- TODO 1: SELECT Artist.ArtistName, Concert.VenueName FROM ... JOIN ... ON ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [39]:
check_rows("TODO 1", "q1", [("Dana Voss", "Riverside Hall"), ("Dana Marsh", "Riverside Hall"), ("The Cobalt Keys", "Blue Note Club"), ("Reo Vantage", "Grand Arena"), ("Dana Voss", "Blue Note Club")])

✗ TODO 1 is not right yet.
   You returned 1 row(s): [('write your SQL here',)]
   Expected     5 row(s): [('Dana Voss', 'Riverside Hall'), ('Dana Marsh', 'Riverside Hall'), ('The Cobalt Keys', 'Blue Note Club'), ('Reo Vantage', 'Grand Arena'), ('Dana Voss', 'Blue Note Club')]


**TODO 2.** Find every artist whose name starts with `The` (select `ArtistName`). Use `LIKE`.

In [27]:
%%sql q2 <<
-- TODO 2: SELECT ArtistName FROM Artist WHERE ArtistName LIKE ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [28]:
check_rows("TODO 2", "q2", [("The Cobalt Keys",)])

✗ TODO 2 is not right yet.
   You returned 1 row(s): [('write your SQL here',)]
   Expected     1 row(s): [('The Cobalt Keys',)]


**TODO 3.** List every concert with a ticket price between 40 and 80 (`ConcertID`, `TicketPrice`), most expensive first.

In [29]:
%%sql q3 <<
-- TODO 3: WHERE ... BETWEEN ... AND ... then ORDER BY ... DESC
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [30]:
check_rows("TODO 3", "q3", [("C2", 60.0), ("C5", 50.0), ("C1", 45.0)], ordered=True)

✗ TODO 3 is not right yet.
   You returned 1 row(s): [('write your SQL here',)]
   Expected     3 row(s): [('C2', 60.0), ('C5', 50.0), ('C1', 45.0)]


**TODO 4.** Find the venues that have hosted **more than one** concert (`VenueName`, and the count). Which clause can test a count?

In [31]:
%%sql q4 <<
-- TODO 4: GROUP BY VenueName, then HAVING COUNT(*) > 1
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [32]:
check_rows("TODO 4", "q4", [("Riverside Hall", 2), ("Blue Note Club", 2)])

✗ TODO 4 is not right yet.
   You returned 1 row(s): [('write your SQL here',)]
   Expected     2 row(s): [('Riverside Hall', 2), ('Blue Note Club', 2)]


**TODO 5 (challenge).** Show each genre with the number of concerts played by artists in that genre (`Genre`, count). You need a JOIN **and** a GROUP BY.

In [33]:
%%sql q5 <<
-- TODO 5: join Artist and Concert, then GROUP BY Genre
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [34]:
check_rows("TODO 5", "q5", [("Indie", 3), ("Jazz", 1), ("Rock", 1)])

✗ TODO 5 is not right yet.
   You returned 1 row(s): [('write your SQL here',)]
   Expected     3 row(s): [('Indie', 3), ('Jazz', 1), ('Rock', 1)]


## 5. Check your understanding

A query needs to find only the venues that have hosted **more than one** concert. Which clause makes that filter possible? **A** `WHERE COUNT(*) > 1`, **B** `HAVING COUNT(*) > 1`, **C** `ORDER BY COUNT(*) > 1`, **D** they both work identically here.

In [35]:
choice = "?"   # type A, B, C or D
check_choice(choice, "B", "WHERE filters individual rows before any grouping. HAVING filters groups after GROUP BY, and it can test an aggregate like COUNT(*).",
             "WHERE runs first, on single rows, so COUNT(*) does not exist yet. HAVING runs after GROUP BY and is the clause that can test an aggregate.")

Set choice to A, B, C or D, then run this cell again.


**Examiner note.** Weak answer: `WHERE COUNT(*) > 1`. That is invalid because `COUNT(*)` does not exist yet when WHERE runs. Better: `GROUP BY VenueName HAVING COUNT(*) > 1`. Group the rows by venue first, **then** filter which groups survive.

## 6. Revision questions

The three queries above (TODO 1 to 4) are the construct questions. Here is the written one.

**Q. Explain why BETWEEN is inclusive of both boundary values, using `TicketPrice BETWEEN 40 AND 80` as your example.** [3 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

## 7. Linking question (connects to B2)

*How do programming languages interact with databases?* A JOIN query rarely runs on its own. A real application embeds it inside Python or Java, which sends the SQL string to the database, waits for the result set and then loops over it like any other collection. The SQL stays declarative (**what** to retrieve) and the surrounding program stays imperative (**how** to use what comes back). In this notebook, `db.execute(...).fetchall()` is exactly that.